In [ ]:
import pandas as pd
import numpy as np
import math
import random
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
import os
import gdown

In [ ]:
# 1. zip 파일의 ID 입력
zip_file_id = 'YOUR_DRIVE_FILE_ID'
output_filename = 'data.zip'

# 2. zip 파일 다운로드
url = f'https://drive.google.com/uc?id={zip_file_id}'
gdown.download(url, output_filename, quiet=False)

# 3. 압축 풀기 (리눅스 명령어를 사용합니다)
# -o: 덮어쓰기 허용, -d: 압축 풀 폴더 지정
!unzip -o data.zip

# 4. 확인
print("파일 목록:", os.listdir())

train = pd.read_csv('data/train.csv', encoding='utf-8-sig')
test = pd.read_csv('data/test.csv', encoding='utf-8-sig')
submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')

In [ ]:
# ==========================================
# 1. 자모 유틸리티 & 정렬 함수
# ==========================================
BASE_CODE = 0xAC00
CHOSUNG  = 21 * 28
JUNGSUNG = 28
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
        else:
            cho  = code // CHOSUNG
            jung = (code % CHOSUNG) // JUNGSUNG
            jong = (code % CHOSUNG) % JUNGSUNG
            result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            cho_i = CHOSUNG_LIST.index(cho)
            jung_i = JUNGSUNG_LIST.index(jung)
            jong_i = JONGSUNG_LIST.index(jong)
            code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    # ML 학습 데이터 라벨링을 위한 정렬 (Levenshtein)
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    
    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], '<DEL>')); i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1])); j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1])); i-=1; j-=1
    return alignment[::-1]

# ==========================================
# 2. ML 데이터셋 생성 (Sliding Window)
# ==========================================
def create_ml_dataset(df, window_size=7):
    data_X = []
    data_y = []
    PAD = '<PAD>'
    
    print("ML 데이터셋 생성 중 (No Augmentation)...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        in_jamos = to_jamo_list(row['input'])
        out_jamos = to_jamo_list(row['output'])
        
        # 정렬을 통해 정답 라벨 생성
        alignment = align_jamo_list(in_jamos, out_jamos)
        
        padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
        current_input_idx = 0
        
        for src_char, tgt_char in alignment:
            if src_char is None: continue # 삽입된 문자는 스킵
            
            # 문맥 윈도우 추출
            window = padded_in[current_input_idx : current_input_idx + window_size]
            data_X.append(window)
            data_y.append(tgt_char)
            current_input_idx += 1
            
    return data_X, data_y

# ==========================================
# 3. 보조 언어 모델 (Language Model)
# ==========================================
def build_language_model(train_df):
    print("통계적 언어 모델(LM) 구축 중...")
    # 정답 데이터의 자모 연결성을 학습하여 ML의 확률을 보정함
    lm_counter = Counter()
    for txt in train_df['output']:
        jamos = [(None)] + to_jamo_list(txt)
        for i in range(len(jamos)-1):
            lm_counter[(jamos[i], jamos[i+1])] += 1
            
    lm_prob = defaultdict(lambda: 1e-6) # 기본값 아주 작게
    total_counts = defaultdict(int)
    for (prev, curr), count in lm_counter.items():
        total_counts[prev] += count
        
    for (prev, curr), count in lm_counter.items():
        lm_prob[(prev, curr)] = count / total_counts[prev]
        
    return lm_prob

# ==========================================
# 4. 하이브리드 디코더 (ML Prob + Beam Search + Re-ranking)
# ==========================================
def decode_hybrid(text, model, token_to_id, id_to_token, classes, lm_prob, vocab_set, 
                  window_size=7, beam_size=20, lm_weight=1.5, threshold=3.0):
    
    in_jamos = to_jamo_list(text)
    PAD = '<PAD>'
    padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
    
    # 1. ML 모델로 전체 시퀀스 확률 예측 (Batch Inference)
    windows = []
    for i in range(len(in_jamos)):
        windows.append(padded_in[i : i + window_size])
    
    unknown_id = token_to_id.get('<PAD>', 0)
    X_enc = [[token_to_id.get(t, unknown_id) for t in w] for w in windows]
    
    # predict_proba: [N_chars, N_classes]
    probs = model.predict_proba(X_enc)
    
    # 2. Beam Search 시작
    # State: (decoded_jamo_list, last_char, score)
    beam = [([], None, 0.0)] 
    
    for i in range(len(in_jamos)):
        curr_probs = probs[i]
        # 속도를 위해 상위 K개 확률만 고려
        top_k_indices = np.argsort(curr_probs)[-beam_size:] 
        
        new_beam = []
        for (decoded, last_char, score) in beam:
            for idx in top_k_indices:
                char_id = classes[idx] # 클래스 ID
                char_prob = curr_probs[idx]
                pred_char = id_to_token[char_id]
                
                # 점수 1: ML 모델의 확신도 (Log Probability)
                ml_score = math.log(char_prob + 1e-9)
                
                # 점수 2: 언어 모델의 자연스러움 (LM Score)
                # 앞 글자(last_char) 다음에 이 글자(pred_char)가 올 확률
                lm_p = lm_prob.get((last_char, pred_char), 1e-6)
                lm_score = math.log(lm_p)
                
                # 최종 점수 결합 (가중치 적용)
                total_score = score + ml_score + (lm_weight * lm_score)
                
                if pred_char == '<DEL>':
                    new_beam.append((decoded, last_char, total_score))
                else:
                    new_beam.append((decoded + [pred_char], pred_char, total_score))
        
        # 가지치기 (Pruning)
        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]
    
    # 3. Re-ranking (사전 기반 교정) - 필수 치트키
    candidates = []
    for (decoded, _, sc) in beam:
        word = from_jamo_list(decoded)
        candidates.append((word, sc))
        
    if not candidates: return text
    
    best_word, best_score = candidates[0]
    if best_word in vocab_set: return best_word
    
    for i in range(1, len(candidates)):
        cand_word, cand_score = candidates[i]
        if best_score - cand_score > threshold: break
        if cand_word in vocab_set: return cand_word
            
    return best_word


In [ ]:
# ==========================================
# 5. 메인 실행 프로세스
# ==========================================

# 1) ML 데이터셋 생성 (증강 X)
X_train, y_train = create_ml_dataset(train, window_size=7)

# 2) 인코딩 & 학습
all_tokens = set(np.array(X_train).flatten()) | set(y_train) | {'<PAD>', '<DEL>'}
token_to_id = {t: i for i, t in enumerate(sorted(list(all_tokens)))}
id_to_token = {i: t for t, i in token_to_id.items()}

# 학습 데이터 인코딩
unknown_id = token_to_id['<PAD>']
X_enc = np.array([[token_to_id.get(t, unknown_id) for t in row] for row in X_train], dtype=np.int16)
y_enc = np.array([token_to_id[t] for t in y_train], dtype=np.int16)

print("RandomForest 학습 시작...")
# n_estimators를 100으로 설정 (시간/성능 타협)
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=1, max_depth=20, random_state=42, verbose=1)
rf_model.fit(X_enc, y_enc)

# 3) 보조 LM & 사전 구축
lm_prob = build_language_model(train)
vocab_set = set()
for txt in train['output']: 
    for w in txt.split(): vocab_set.add(w)

# 4) 추론 (Hybrid)
print("하이브리드 추론 시작 (ML + Beam + Re-ranking)...")
outputs = []
# 모델의 클래스 목록 미리 저장
rf_classes = rf_model.classes_

for idx, row in tqdm(test.iterrows(), total=len(test)):
    obf_text = row['input']
    tokens = obf_text.split()
    new_tokens = []
    
    for t in tokens:
        restored = decode_hybrid(t, rf_model, token_to_id, id_to_token, rf_classes, lm_prob, vocab_set,
                                 window_size=7, beam_size=20, lm_weight=1.5, threshold=3.0)
        new_tokens.append(restored)
    outputs.append(" ".join(new_tokens))

sample_submission['output'] = outputs
sample_submission.to_csv('submission_final.csv', index=False)
print("완료! 저장되었습니다.")